# mup integration with perspic

This notebook walks through how to integrate **Maximal Update Parametrization (mup)** with the **perspic analyzer**, so you can study scaling behaviour of your models.

[mup](https://github.com/microsoft/mup) is a parametrization scheme that lets you transfer hyperparameters (especially the learning rate) from a small "base" model to a much wider "target" model without re-tuning. The idea is to mark certain dimensions as **width** dimensions and apply width-dependent scaling to learning rates and to the output layer. This is exactly the kind of behaviour that perspic is designed to inspect.

If perspic is installed, mup is already available in your environment. Otherwise install it manually:

```
pip install mup
```

**What you need to change in your code.** There are essentially three things:

1. **Replace the readout layer.** Any `nn.Linear` that maps from a width dimension to a fixed output dimension (e.g. `num_classes`, `vocab_size`) must become a `MuReadout`. Because of the way perspic computes per-sample gradients, we use a small subclass `CachedMuReadout` (explained in section 1).
2. **Use a mup optimizer.** Replace `torch.optim.Adam` (or `SGD`) with `MuAdam` (or `MuSGD`) inside `configure_optimizers` of your `LightningModule`.
3. **Call `set_base_shapes`.** Build a small "base" version of your model in addition to the target one and pass both to `set_base_shapes(model, base)` *before* training. This is what tells mup which dimensions are widths.

The rest of this notebook explains each step in turn, runs a small end-to-end perspic example with a tiny transformer, and then shows the concrete code changes for three real architectures: a `WideResNet`, the torchvision `VisionTransformer`, and a Llama-style transformer.

## 1.) Add modified readout layer to your model

In [3]:
from mup import MuReadout
class CachedMuReadout(MuReadout):
    """MuReadout that auto-caches width_mult for functorch compatibility."""

    def forward(self, x):
        if not hasattr(self, '_cached_width_mult'):
            # First call happens during normal forward pass, infshape exists
            self._cached_width_mult = self.weight.infshape.width_mult()
        return nn.Linear.forward(self, self.output_mult * x / self._cached_width_mult)

**Why a custom `CachedMuReadout` instead of `MuReadout`?**

`MuReadout.forward()` accesses `self.weight.infshape` on every call. This breaks under perspic's per-sample gradient computation, which uses opacus + functorch internally.

When opacus falls back to functorch for layers without a registered grad sampler (like `MuReadout`), it calls `torch.func.functional_call`, which **temporarily replaces `self.weight` with a plain tensor** during the backward pass. That replacement tensor does **not** carry the `.infshape` attribute, so `MuReadout.forward()` raises `AssertionError: Please call set_base_shapes(...)`.

The fix is to cache `width_mult()` as a regular Python attribute on the first normal forward pass (when `.infshape` is still present). On subsequent calls — including the functorch-reparametrized ones — the cached value is used and `.infshape` is never touched. This is safe because `width_mult` is static metadata determined entirely by the model's architecture; it never changes during training.

## 2.) Use a mup optimizer in your LightningModule

Mup applies a per-parameter learning-rate scaling that depends on each parameter's `infshape`. This scaling is implemented inside the optimizer, so you must use one of the mup-aware variants instead of the standard PyTorch ones:

- `MuAdam` — drop-in replacement for `torch.optim.Adam`
- `MuAdamW` — drop-in replacement for `torch.optim.AdamW`
- `MuSGD` — drop-in replacement for `torch.optim.SGD`

Inside your `LightningModule`, just override `configure_optimizers` to return one of these:

In [6]:
from mup import MuAdam, MuAdamW, MuSGD
from pytorch_lightning import LightningModule

class LitMuAdam(LightningModule):
    def __init__(self, model, lr=1e-3):
        super().__init__()
        self.model = model
        self.lr = lr

# Override to use MuAdam optimizer instead of the default Adam !! 
    def configure_optimizers(self):
        return MuAdam(self.model.parameters(), lr=self.lr)

## 3.) Using `set_base_shapes`

Once your model has a `CachedMuReadout` and your LightningModule uses `MuAdam`, the only remaining step is to tell mup which dimensions are "width" dimensions. You do this by instantiating **two (or three) versions of the same model** with different widths and calling `set_base_shapes`:

```python
from mup import set_base_shapes

base  = MyModel(width=64)    # smallest
model = MyModel(width=256)   # target training width

set_base_shapes(model, base)  # marks dimensions that differ as "infinite"
```

`set_base_shapes` walks every parameter in `base` and `model`, compares their shapes, and marks any dimension that **changes** between the two as a width dimension (infinite). Dimensions that stay the same are marked finite. Optionally, you can pass a third `delta` model to disambiguate edge cases — but for most architectures, base + target is enough.

**Important:** the base and target models must have **identical structure** — same layer names, same number of layers. Only the width-related dimensions should differ. Structural choices like `num_layers`, `patch_size`, `image_size`, `num_classes` must match across all models.

Below is a minimal end-to-end example using the `TinyMuPTransformer` from earlier together with the perspic analyzer:

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.utils.data import TensorDataset, DataLoader
from mup import MuAdam, set_base_shapes
import perspic


class TinyMuPTransformer(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, nhead=4, nlayers=2, max_len=64):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.01)
        # NOTE: dim_feedforward must scale with d_model — see "Problem 1" callout below
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.out = CachedMuReadout(d_model, vocab_size)

    def forward(self, x):
        h = self.tok(x) + self.pos[:, : x.size(1), :]
        h = self.encoder(h)
        return self.out(h[:, -1, :])  # [B, V] — last token only (perspic expects 2D)


class LanguageModelingModule(pl.LightningModule):
    def __init__(self, model, lr=1e-3):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.criterion = lambda logits, targets: F.cross_entropy(logits, targets[:, -1])

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return MuAdam(self.parameters(), lr=self.hparams.lr)


# Create base + target and set mup shapes
base  = TinyMuPTransformer(d_model=64)
model = TinyMuPTransformer(d_model=128)
set_base_shapes(model, base)

# Wrap with perspic analyzer
lit_model = perspic.analyzer(
    lightning_module=LanguageModelingModule,
    model=model,
    lr=1e-3,
)

# Dummy data
x = torch.randint(0, 1000, (256, 32))
y = torch.randint(0, 1000, (256, 32))
train_loader = DataLoader(TensorDataset(x, y), batch_size=8)

trainer = pl.Trainer(max_epochs=-1, max_steps=5, accelerator="auto", devices=1)
trainer.fit(lit_model, train_dataloaders=train_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type               | Params | Mode 
-----------------------------------------------------
0 | model | TinyMuPTransformer | 661 K  | train
-----------------------------------------------------
661 K     Trainable params
0         Non-trainable params
661 K     Total params
2.647     Total estimated model params size (MB)
25        Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=5` reached.


> **⚠️ Problem: Every internal width-dependent dimension must scale**
>
> Notice the line `dim_feedforward=d_model * 4` above. By default, `nn.TransformerEncoderLayer` uses `dim_feedforward=2048`, a fixed value. If `d_model` varies across base/target but `dim_feedforward` stays at 2048, then `linear1` (the first FFN layer) has shape `[2048, d_model]` — fan-out is **finite** (2048 in both models) but fan-in is **infinite** (varies). MuP requires any layer with infinite fan-in and finite fan-out to be a `MuReadout`, and raises:
>
> ```
> AssertionError: encoder.layers.0.linear1 has infinite fan-in and finite fan-out
> dimensions but is not type `MuReadout`.
> ```
>
> **Rule:** every internal dimension that depends on width must scale proportionally with width. Fixed-size hidden dimensions, projection sizes, or bottleneck widths between two width-scaled layers will all trip this check. Either scale them with width, or replace the offending layer with `MuReadout`.

#### Check if the .infshape attributes are registrated

In [10]:
print("\nAfter analyzer (lit_model.model):")
for name, p in lit_model.model.named_parameters():
    has_inf = hasattr(p, "infshape")
    if "out" in name:
        print(f"  {name}: infshape={has_inf}")


After analyzer (lit_model.model):
  encoder.layers.0.self_attn.out_proj.weight: infshape=True
  encoder.layers.0.self_attn.out_proj.bias: infshape=True
  encoder.layers.1.self_attn.out_proj.weight: infshape=True
  encoder.layers.1.self_attn.out_proj.bias: infshape=True
  out.weight: infshape=True
  out.bias: infshape=True


## 4.) Modifications for real-world architectures

The recipe is always the same:
1. Find the **final readout layer** (width → fixed output) and replace it with `CachedMuReadout`.
2. Find any **internal layer with a fixed width dimension** and make it scale with width.
3. Make sure the base and target models have **identical structure** — only width dimensions should differ.

Below are the concrete changes for three common architectures: a `WideResNet`, the torchvision `VisionTransformer`, and a Llama-style transformer.

### 4a) WideResNet

In `WideResNet`, the width is controlled by `widen_factor`. The channel counts inside the network are `[16, widen_factor, 2*widen_factor, 4*widen_factor]`. The only layer that maps a width dimension to a fixed output is the final classifier `self.fc`:

```python
# Before
self.fc = nn.Linear(nChannels[3], num_classes)

# After
self.fc = CachedMuReadout(nChannels[3], num_classes)
```

All convolutions are internal (channels → channels) — they need no changes. Mup handles them automatically via `MuAdam`'s width-dependent learning-rate scaling.

**Caveat — `convShortcut`:** the residual block only creates a `convShortcut` when `in_planes != out_planes`. If your base model happens to use `widen_factor == nChannels[0]` (i.e. `widen_factor=16`), the very first block has `in_planes == out_planes == 16` and **no** shortcut conv, but a larger target model has `in_planes=16, out_planes=widen_factor != 16` and **does** have one. The parameter names then differ between base and target, and `set_base_shapes` will fail with:

```
AssertionError: `shapes` has extra names {'block1.layer.0.convShortcut.weight'}
```

Fix: never use `widen_factor=16` (or whatever value equals `nChannels[0]`) for *any* of base/target. Pick something like `widen_factor=24` for the base and `widen_factor=32` for the target:

```python
base  = WideResNet(depth=10, num_classes=10, widen_factor=24)
model = WideResNet(depth=10, num_classes=10, widen_factor=32)
set_base_shapes(model, base)
```

The cell below contains a complete WideResNet implementation with the single `nn.Linear → CachedMuReadout` swap already applied. You can copy it into your own project as a starting point.


In [12]:
# Full mup-ready WideResNet — only change vs. the standard implementation
# is the final classifier: nn.Linear -> CachedMuReadout.
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropRate=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes, momentum=0.1)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes, momentum=0.1)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = in_planes == out_planes
        self.convShortcut = (
            (not self.equalInOut)
            and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, padding=0, bias=False)
            or None
        )

    def forward(self, x):
        if not self.equalInOut:
            x = self.relu1(self.bn1(x))
        else:
            out = self.relu1(self.bn1(x))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
        if self.droprate > 0:
            out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.equalInOut else self.convShortcut(x), out)


class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super().__init__()
        layers = []
        for i in range(int(nb_layers)):
            layers.append(block(i == 0 and in_planes or out_planes, out_planes, i == 0 and stride or 1, dropRate))
        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


class WideResNet(nn.Module):
    def __init__(self, depth, num_classes, widen_factor=16, dropRate=0.0):
        super().__init__()
        nChannels = [16, widen_factor, 2 * widen_factor, 4 * widen_factor]
        assert (depth - 4) % 6 == 0
        n = (depth - 4) / 6
        self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], BasicBlock, 1, dropRate)
        self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], BasicBlock, 2, dropRate)
        self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], BasicBlock, 2, dropRate)
        self.bn1 = nn.BatchNorm2d(nChannels[3])
        self.relu = nn.ReLU(inplace=True)

        # === MuP change: readout is CachedMuReadout instead of nn.Linear ===
        self.fc = CachedMuReadout(nChannels[3], num_classes)

        self.nChannels = nChannels[3]

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.avg_pool2d(out, 8)
        out = out.view(-1, self.nChannels)
        return self.fc(out)


# Example usage with set_base_shapes — note: avoid widen_factor=16 (== nChannels[0])
# base  = WideResNet(depth=10, num_classes=10, widen_factor=24)
# model = WideResNet(depth=10, num_classes=10, widen_factor=32)
# set_base_shapes(model, base)
# apply perspic.analyzer with a LightningModule wrapper like above.

### 4b) torchvision `VisionTransformer`

`torchvision.models.VisionTransformer` creates the final classifier internally as `self.heads.head = nn.Linear(hidden_dim, num_classes)`. You can't pass a custom layer to the constructor, so the cleanest fix is to **replace the layer after construction**:

```python
from torchvision.models import VisionTransformer

def make_vit(hidden_dim, mlp_dim, num_heads):
    vit = VisionTransformer(
        image_size=32, patch_size=4,
        num_layers=6,            # MUST be the same across base and target
        num_heads=num_heads,
        hidden_dim=hidden_dim,
        mlp_dim=mlp_dim,
        num_classes=10,          # MUST be the same across base and target
    )
    vit.heads.head = CachedMuReadout(hidden_dim, 10)
    return vit

base  = make_vit(hidden_dim=128, mlp_dim=256, num_heads=4)
model = make_vit(hidden_dim=256, mlp_dim=512, num_heads=8)
set_base_shapes(model, base)
```

**Pitfalls to watch for:**
- `num_layers` must be identical across base and target. Otherwise base has missing parameter names and `set_base_shapes` fails.
- `num_classes`, `image_size`, `patch_size` are structural — keep them constant.
- Width dimensions to vary across base and target: `hidden_dim`, `mlp_dim`, `num_heads` (or equivalently `head_dim`).

### 4c) Llama-style transformer

For a Llama-style model, the width dimension is `args.dim`. The final readout is `self.output = nn.Linear(args.dim, args.vocab_size, bias=False)` — replace it with `CachedMuReadout`:

```python
# Before:  self.output = nn.Linear(args.dim, args.vocab_size, bias=False)
# After:
self.output = CachedMuReadout(args.dim, args.vocab_size, bias=False)
```

**Other things to check inside the Llama internals:**

1. **`Attention`**: the projections `wq, wk, wv, wo` map between `args.dim` and `n_heads * head_dim`. Both sides scale with width as long as `head_dim = args.dim // args.n_heads` — which is already the case in the standard Llama implementation. ✅ No changes needed.
2. **`FeedForward`**: the hidden dim is computed as `hidden_dim = args.multiple_of * ceil(8/3 * args.dim / args.multiple_of)`. This already scales with `args.dim`. ✅ No changes needed. (Compare with the `dim_feedforward=2048` problem in `TinyMuPTransformer` from section 3 — that was a bug because the FFN width was hard-coded; in Llama it is derived from `args.dim` and so is fine.)
3. **Weight tying**: the line `self.output.weight = self.tok_embeddings.weight` must stay disabled — perspic's per-sample gradient computation does not support tied weights (opacus's ghost clipping raises `NotImplementedError` for parameter tying).
4. **Output shape**: `self.output(x)` returns `[B, T, vocab_size]` (3D). Perspic's analyzer expects 2D `[B, C]`, so you must either pool the sequence dimension before the readout or — for next-token prediction — take just the last position:

   ```python
   def forward(self, idx):
       bsz, seqlen = idx.shape
       x = self.tok_embeddings(idx)
       freqs_cis = self.freqs_cis[:seqlen].to(x.device)
       for layer in self.layers:
           x = layer(x, freqs_cis)
       x = self.norm(x)
       return self.output(x[:, -1, :])  # [B, vocab_size] — last token only
   ```

   And in the LightningModule, slice the targets to match: `self.criterion = lambda logits, targets: F.cross_entropy(logits, targets[:, -1])`.

5. **Structural constants across base/target**: `n_layers`, `vocab_size`, `max_seq_len`, `multiple_of`, `n_kv_heads` must all be identical. Width dimensions to vary across base and target: `dim`, `n_heads` (keeping `dim // n_heads` constant gives the cleanest mup comparison).

```python
base_args  = ModelArgs(dim=128, n_heads=4,  n_layers=4, vocab_size=1000, max_seq_len=64)
model_args = ModelArgs(dim=512, n_heads=16, n_layers=4, vocab_size=1000, max_seq_len=64)

base  = Transformer(base_args)
model = Transformer(model_args)
set_base_shapes(model, base)
```


The cell below contains a complete Llama-style transformer with the readout swap and the `forward()` change to last-token prediction already applied. You can copy it into your own project as a starting point.

In [14]:
# Full mup-ready Llama-style transformer.
# Changes vs. the standard implementation:
#   1. self.output is CachedMuReadout instead of nn.Linear
#   2. forward() returns the last-token logits ([B, vocab_size]) so perspic's
#      analyzer (which expects 2D output) can compute per-sample gradients.
#   3. Weight tying (self.output.weight = self.tok_embeddings.weight) stays disabled.
import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelArgs:
    dim: int = 512
    n_layers: int = 4
    n_heads: int = 8
    n_kv_heads: Optional[int] = None
    vocab_size: int = 1000
    multiple_of: int = 32
    norm_eps: float = 1e-5
    max_seq_len: int = 64
    dropout: float = 0.0


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return self._norm(x.float()).type_as(x) * self.weight


def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    return torch.polar(torch.ones_like(freqs), freqs)


def reshape_for_broadcast(freqs_cis, x):
    ndim = x.ndim
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)


def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)


class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        self.head_dim = args.dim // args.n_heads

        # All these projections are width -> width — no mup change needed.
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)
        self.dropout = args.dropout

    def forward(self, x, freqs_cis):
        bsz, seqlen, _ = x.shape
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_kv_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_kv_heads, self.head_dim)
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis)
        out = F.scaled_dot_product_attention(
            xq.transpose(1, 2), xk.transpose(1, 2), xv.transpose(1, 2),
            is_causal=True, dropout_p=self.dropout if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        return self.wo(out)


class FeedForward(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        # hidden_dim is derived from args.dim, so it scales with width — no fix needed.
        hidden_dim = 4 * args.dim
        hidden_dim = int(2 * hidden_dim / 3)
        hidden_dim = args.multiple_of * ((hidden_dim + args.multiple_of - 1) // args.multiple_of)
        self.w1 = nn.Linear(args.dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, args.dim, bias=False)
        self.w3 = nn.Linear(args.dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # SwiGLU


class TransformerBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.attention = Attention(args)
        self.feed_forward = FeedForward(args)
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)

    def forward(self, x, freqs_cis):
        h = x + self.attention(self.attention_norm(x), freqs_cis)
        return h + self.feed_forward(self.ffn_norm(h))


class Transformer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
        self.layers = nn.ModuleList([TransformerBlock(args) for _ in range(args.n_layers)])
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)

        # === MuP change: readout is CachedMuReadout instead of nn.Linear ===
        self.output = CachedMuReadout(args.dim, args.vocab_size, bias=False)

        # Weight tying disabled — opacus ghost clipping cannot handle tied weights.
        # self.output.weight = self.tok_embeddings.weight

        freqs_cis = precompute_freqs_cis(
            self.args.dim // self.args.n_heads, self.args.max_seq_len * 2
        )
        self.register_buffer("freqs_cis", freqs_cis, persistent=False)

    def forward(self, idx):
        bsz, seqlen = idx.shape
        x = self.tok_embeddings(idx)
        freqs_cis = self.freqs_cis[:seqlen].to(x.device)
        for layer in self.layers:
            x = layer(x, freqs_cis)
        x = self.norm(x)
        # === MuP/perspic change: return last-token logits, shape [B, vocab_size] ===
        return self.output(x[:, -1, :])


# Example usage with set_base_shapes
# base_args  = ModelArgs(dim=128, n_heads=4,  n_layers=4, vocab_size=1000, max_seq_len=64)
# model_args = ModelArgs(dim=512, n_heads=16, n_layers=4, vocab_size=1000, max_seq_len=64)
# base  = Transformer(base_args)
# model = Transformer(model_args)
# set_base_shapes(model, base)
# apply perspic.analyzer with a LightningModule wrapper like above.